# DenseNet Model
Bu notebook DenseNet modelinin eğitimi ve değerlendirmesi için oluşturulmuştur.

1. Import

In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

2. Dataset Paths

In [ ]:
base_path = "../data/"

train_dir = os.path.join(base_path, "train")
valid_dir = os.path.join(base_path, "valid")
test_dir = os.path.join(base_path, "test")

print("Train:", train_dir)
print("Valid:", valid_dir)
print("Test:", test_dir)

print("Train classes:", os.listdir(train_dir))
print("Valid classes:", os.listdir(valid_dir))
print("Test classes:", os.listdir(test_dir))

3. Transform

In [ ]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(7),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

valid_test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

4. Dataset

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
valid_dataset = datasets.ImageFolder(valid_dir, transform=valid_test_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=valid_test_transforms)

print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))
print("Test size:", len(test_dataset))

5. DataLoader

In [ ]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("Test batches:", len(test_loader))

6. Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

7. DenseNet Model

In [ ]:
densenet_model = models.densenet121(weights=None)

densenet_model.features.conv0 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    2
)

densenet_model = densenet_model.to(device)

print("DenseNet121 model is ready.")

8. Loss, Optimizer, Epoch

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    densenet_model.parameters(),
    lr=0.0001
)

num_epochs = 5

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

11. Training Code

In [ ]:
densenet_train_losses = []
densenet_valid_losses = []

densenet_train_accuracies = []
densenet_valid_accuracies = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        densenet_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_acc = validate(
        densenet_model,
        valid_loader,
        criterion,
        device
    )

    densenet_train_losses.append(train_loss)
    densenet_valid_losses.append(valid_loss)

    densenet_train_accuracies.append(train_acc)
    densenet_valid_accuracies.append(valid_acc)

    print(f"\nDenseNet121 Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Accuracy: {valid_acc:.2f}%")

12. Save Model

In [ ]:
os.makedirs("../models", exist_ok=True)

torch.save(
    densenet_model.state_dict(),
    "../models/densenet121_chest_xray.pth"
)

print("DenseNet121 model saved successfully.")

13. Load Saved DenseNet Model

In [ ]:
densenet_model = models.densenet121(weights=None)

densenet_model.features.conv0 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    2
)

densenet_model.load_state_dict(
    torch.load(
        "../models/densenet121_chest_xray.pth",
        map_location=device
    )
)

densenet_model = densenet_model.to(device)
densenet_model.eval()

print("Saved DenseNet121 model loaded successfully.")

In [ ]:
os.makedirs("../outputs", exist_ok=True)

plt.figure(figsize=(12, 5))

# Loss graph
plt.subplot(1, 2, 1)

plt.plot(densenet_train_losses, label="Train Loss")
plt.plot(densenet_valid_losses, label="Validation Loss")

plt.title(f"DenseNet121 {num_epochs} Epoch Loss Analysis")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

# Accuracy graph
plt.subplot(1, 2, 2)

plt.plot(densenet_train_accuracies, label="Train Accuracy")
plt.plot(densenet_valid_accuracies, label="Validation Accuracy")

plt.title(f"DenseNet121 {num_epochs} Epoch Accuracy Analysis")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()

plt.savefig(f"../outputs/densenet121_training_analysis_{num_epochs}epoch.png")

plt.show()

14. DenseNet Evaluation

In [ ]:
densenet_all_labels = []
densenet_all_preds = []
densenet_all_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = densenet_model(images)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        densenet_all_probs.extend(probs[:, 1].cpu().numpy())
        densenet_all_preds.extend(preds.cpu().numpy())
        densenet_all_labels.extend(labels.cpu().numpy())

densenet_all_probs = np.array(densenet_all_probs)
densenet_all_preds = np.array(densenet_all_preds)
densenet_all_labels = np.array(densenet_all_labels)

print("DenseNet121 evaluation completed.")

15. Metrics

In [ ]:
densenet_accuracy = accuracy_score(densenet_all_labels, densenet_all_preds)
densenet_precision = precision_score(densenet_all_labels, densenet_all_preds)
densenet_recall = recall_score(densenet_all_labels, densenet_all_preds)
densenet_f1 = f1_score(densenet_all_labels, densenet_all_preds)
densenet_auc = roc_auc_score(densenet_all_labels, densenet_all_probs)

print(f"Accuracy : {densenet_accuracy:.4f}")
print(f"Precision: {densenet_precision:.4f}")
print(f"Recall   : {densenet_recall:.4f}")
print(f"F1-Score : {densenet_f1:.4f}")
print(f"ROC-AUC  : {densenet_auc:.4f}")

16. Confusion Matrix

In [ ]:
cm = confusion_matrix(
    densenet_all_labels,
    densenet_all_preds
)

tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("DenseNet121 Confusion Matrix")

plt.savefig("../outputs/densenet121_confusion_matrix.png")
plt.show()

print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

17. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(
    densenet_all_labels,
    densenet_all_probs
)

plt.figure(figsize=(7, 6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {densenet_auc:.4f}"
)

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("DenseNet121 ROC Curve")

plt.legend()

plt.savefig("../outputs/densenet121_roc_curve.png")
plt.show()

In [ ]:

import pandas as pd

os.makedirs("../outputs/model_results", exist_ok=True)

densenet_results_df = pd.DataFrame({
    "Model": ["DenseNet121"],
    "Epoch": [num_epochs],
    "Best Validation Accuracy": [
        max(densenet_valid_accuracies) if len(densenet_valid_accuracies) > 0 else np.nan
    ],
    "Final Validation Accuracy": [
        densenet_valid_accuracies[-1] if len(densenet_valid_accuracies) > 0 else np.nan
    ],
    "Test Accuracy": [densenet_accuracy * 100],
    "Precision": [densenet_precision],
    "Recall": [densenet_recall],
    "F1-Score": [densenet_f1],
    "ROC-AUC": [densenet_auc]
})

densenet_results_df.to_csv(
    "../outputs/model_results/densenet121_results.csv",
    index=False
)

densenet_results_df

### DenseNet121 Evaluation Summary

DenseNet121 was implemented as an advanced deep learning architecture for chest X-ray classification. The model was trained using CLAHE-preprocessed grayscale medical images.

Performance evaluation included accuracy, precision, recall, F1-score, confusion matrix analysis, and ROC-AUC measurement. The model demonstrated strong feature extraction capability due to its dense connectivity architecture.

DenseNet121 results were compared with ResNet18 and EfficientNet-B0 models to determine the best-performing architecture for abnormality detection in chest X-ray images.